<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Memory Retrieval Test </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Exercise: Memory Retrieval Test_sol

**Purpose:**  
Validate that long-term memory retrieval improves agent behavior by running multiple interactions and checking if the agent remembers previous context.

**Real-Life Scenario:**  
You are building a **Personal Nutrition Assistant** that helps users with meal planning, ingredient information, dietary restrictions, and nutrition advice. The agent must remember past conversations to provide consistent and personalized recommendations.

## 1. Setup

In [1]:
# Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" faiss-cpu beautifulsoup4 requests --force-reinstall
from datetime import datetime
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.

## 2. Tools

Your agent has these tools for the Nutrition Assistant:

In [2]:
from langchain_core.tools import tool
@tool
def get_nutrition_info(food: str) -> str:
    """Get nutritional information and suggestions for common food items."""
    food = food.lower().strip()
    data = {
        "scrambled tofu": "High in protein (~20g per serving), low in calories, good source of iron and calcium. Great vegetarian breakfast option.",
        "greek yogurt": "High in protein (15-20g per serving), low in sugar if plain. Excellent for weight loss and muscle building.",
        "chia pudding": "High in fiber and omega-3s, provides about 10g protein per serving when made with milk or yogurt.",
        "tofu": "Complete plant-based protein (8-10g per 100g). Versatile for dinner recipes.",
        "quinoa": "Complete protein (14g per 100g), rich in fiber and minerals. Good base for vegetarian meals.",
        "daily calorie intake for weight loss": "For weight loss, a common target is 500-1000 calories below maintenance. Typical range for adults: 1500-2000 calories per day depending on age, gender, and activity level.",
        "vegetarian high protein dinner recipe": "Example: Stir-fried tofu with quinoa, broccoli, and spinach. Provides ~25-30g protein per serving."
    }
    return data.get(food, f"Basic nutritional info for '{food}' is not available in this demo. Try 'scrambled tofu', 'Greek yogurt', or 'tofu'.")

In [3]:
#Long-Term Vector Store Memory (
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(["Initial empty memory"], embeddings)

def store_interaction(query: str, response: str):
    doc = Document(
        page_content=f"User: {query}\nAssistant: {response}",
        metadata={"timestamp": str(datetime.now())}
    )
    vectorstore.add_documents([doc])
    print(f"Stored interaction: {query[:60]}...")

def retrieve_relevant_memory(query: str, k: int = 3):
    docs = vectorstore.similarity_search(query, k=k)
    return "\n\n".join([doc.page_content for doc in docs])

In [5]:
tools = [get_nutrition_info]

In [6]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Use the standard ReAct prompt (no {history} in template)
prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

# Create the agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=12,
    handle_parsing_errors=True
)

print("agent created successfully with standard ReAct prompt")

agent created successfully with standard ReAct prompt


In [7]:
print("=== Multi-turn Conversation with Long-Term Memory ===\n")

queries = [
    "I am vegetarian. Suggest some high-protein breakfast options.",
    "I am trying to lose weight. How many calories should I aim for daily?",
    "Remember I am vegetarian. Suggest a balanced dinner recipe with high protein."
]

for q in queries:
    print(f"\n{'='*80}")
    print(f"User: {q}")

    # Retrieve relevant past memory
    past_memory = retrieve_relevant_memory(q)
    if past_memory:
        print(f"(Retrieved from long-term memory)\n{past_memory}\n")

    # Pass memory into the agent via the {history} slot
    response = executor.invoke({
        "input": q,
        "history": past_memory or "No relevant past interactions found."
    })
    # # Run the agent with the current query
    # response = executor.invoke({"input": q})

    print("Agent:", response["output"])

    # Store this interaction for future retrieval
    store_interaction(q, response["output"])

=== Multi-turn Conversation with Long-Term Memory ===


User: I am vegetarian. Suggest some high-protein breakfast options.
(Retrieved from long-term memory)
Initial empty memory



> Entering new AgentExecutor chain...
I should use the get_nutrition_info tool to find high-protein breakfast options suitable for vegetarians.
Action: get_nutrition_info
Action Input: "vegetarian high-protein breakfast options"Basic nutritional info for 'vegetarian high-protein breakfast options' is not available in this demo. Try 'scrambled tofu', 'Greek yogurt', or 'tofu'.I should try searching for specific vegetarian high-protein breakfast options like scrambled tofu, Greek yogurt, or tofu.
Action: get_nutrition_info
Action Input: "scrambled tofu"High in protein (~20g per serving), low in calories, good source of iron and calcium. Great vegetarian breakfast option.Scrambled tofu is a great high-protein breakfast option for vegetarians.
Final Answer: Scrambled tofu is a high-protein breakfast option for 